[![cloudevel](img/cloudevel.png)](https://cloudevel.com)

# Despliegue en *Cloud Run*.

*Cloud Run* es un servicio de *Google Cloud Platform* (*GCP*) que ejecuta un contenedor de forma gestionada: recibe una imagen, la pone a la escucha de peticiones *HTTP* (*Hypertext Transfer Protocol*) y escala el número de instancias según la demanda.

Este cuaderno hace un **primer despliegue imperativo**: se usa la herramienta de línea de comandos ```gcloud``` y el *workflow* ```.github/workflows/deploy-cloud-run.yml```. No se usa *Terraform* ni se aprovisiona una base de datos gestionada.

## Objetivos.

* Identificar los conceptos de proyecto, cuenta de servicio y roles de *IAM* (*Identity and Access Management*).
* Publicar la imagen de la *API* en *Artifact Registry* y desplegarla en *Cloud Run* con ```gcloud run deploy```.
* Ejecutar el mismo despliegue desde un *workflow*, con un patrón de autenticación transitorio y fácil de sustituir.

## Qué entra en este laboratorio y qué no.

**Sí se practica:** usar un repositorio de *Artifact Registry*, construir y publicar la imagen, desplegar un servicio de *Cloud Run* y comprobar su *URL* (*Uniform Resource Locator*) pública.

**Queda fuera de este curso** (se introduce en cursos posteriores):

* *Cloud SQL* (base de datos relacional gestionada).
* *Secret Manager* (almacén de secretos de *GCP*).
* *Compute Engine* (máquinas virtuales).
* *Terraform* e infraestructura como código.
* La federación de identidad (*Workload Identity Federation*).

Al cierre hay un mapa breve de esos servicios, solo para saber qué son.

## Proyecto, región y *IAM*.

Un **proyecto** de *GCP* agrupa recursos, facturación y permisos. La **región** es la zona geográfica en la que se crea el servicio; la elige quien imparte el curso y se guarda en la variable de repositorio ```GCP_REGION```. No se asume una región concreta en este material.

*IAM* decide quién puede hacer qué. En este laboratorio intervienen, como mínimo:

* El **proyecto**, identificado por la variable de repositorio ```GCP_PROJECT_ID```.
* El **repositorio de *Artifact Registry***, identificado por la variable ```AR_REPOSITORY```.
* El **servicio de *Cloud Run***, identificado por la variable ```CLOUD_RUN_SERVICE```.
* Una **cuenta de servicio** de despliegue, con permisos acotados para publicar imágenes en *Artifact Registry* y actualizar el servicio de *Cloud Run*.
* La **cuenta de servicio de ejecución** del propio servicio de *Cloud Run* (la identidad con la que corre el contenedor), distinta de la que despliega.

Quien imparte el curso entrega esos identificadores y se cargan como variables del repositorio en *GitHub*. Nada de eso se escribe como un valor fijo en el código.

## Autenticación en el laboratorio (valor por defecto).

Hay dos momentos distintos:

1. **En la terminal del *Codespace***, para un despliegue manual de comprobación, se usa el *SDK* de *Google Cloud* (*Software Development Kit*), ya instalado en el contenedor de desarrollo, y ```gcloud auth login```, seguido de la configuración del proyecto.

2. **En *GitHub Actions***, el valor por defecto de este curso es un patrón transitorio de enseñanza: una llave de cuenta de servicio, en formato *JSON* (*JavaScript Object Notation*), que quien imparte el curso carga como secreto del repositorio bajo el nombre ```GCP_SA_KEY```. Cada estudiante trabaja sobre su propio proyecto de *GCP* y su propio repositorio de *GitHub*.


* La siguiente celda configurará el proyecto de *GCP* en el *SDK*, una vez autenticado.


In [ ]:
gcloud config set project "$GCP_PROJECT_ID"

Ese patrón **no** es el adecuado en producción: una llave de larga duración puede exfiltrarse y reutilizarse fuera del *pipeline*. El cuaderno 13 ya presentó *OpenID Connect* (*OIDC*) como la vía recomendada. La práctica de la federación de identidad se deja para cursos posteriores.

El paso de autenticación del *workflow* está aislado, de modo que pueda sustituirse por *Workload Identity Federation* cambiando solo ese paso (y, si aplica, añadiendo ```id-token: write``` a ```permissions```), sin tocar el resto del despliegue.

## Despliegue manual de comprobación.

Con el *SDK* autenticado y las variables de entorno ```GCP_PROJECT_ID```, ```GCP_REGION```, ```AR_REPOSITORY``` y ```CLOUD_RUN_SERVICE``` definidas en la sesión de terminal del *Codespace* (nunca en un archivo versionado):


* La siguiente celda habilitará las *API*, publicará la imagen y desplegará el servicio en *Cloud Run*.


In [ ]:
gcloud services enable run.googleapis.com artifactregistry.googleapis.com

gcloud artifacts repositories create "$AR_REPOSITORY" \
  --repository-format=docker \
  --location="$GCP_REGION" \
  --project="$GCP_PROJECT_ID"

# Si el repositorio ya existe, el comando anterior informa de ello y se puede continuar.

gcloud auth configure-docker "${GCP_REGION}-docker.pkg.dev"

IMAGE="${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT_ID}/${AR_REPOSITORY}/${CLOUD_RUN_SERVICE}:manual"

docker build -t "$IMAGE" .
docker push "$IMAGE"

gcloud run deploy "$CLOUD_RUN_SERVICE" \
  --project="$GCP_PROJECT_ID" \
  --image="$IMAGE" \
  --region="$GCP_REGION" \
  --platform=managed \
  --port=8080 \
  --allow-unauthenticated \
  --set-env-vars "APP_VERSION=manual"

La opción ```--allow-unauthenticated``` deja el servicio accesible de forma pública, que es lo habitual en un laboratorio de primer contacto. Quien imparte el curso puede exigir lo contrario; en ese caso, la comprobación se hace con una identidad autorizada.

Al terminar, ```gcloud run services describe``` muestra la *URL* del servicio. Una petición a ```GET /health``` en esa *URL* debe responder ```"status": "ok"```.

## El *workflow* ```deploy-cloud-run.yml```.

El archivo vigente es ```.github/workflows/deploy-cloud-run.yml```. Se dispara de forma manual (*workflow_dispatch*) o al empujar una etiqueta que empiece por ```v``` (por ejemplo, una etiqueta de versión). Usa *runners* hospedados.


* La siguiente celda mostrará los pasos esenciales, alineados con ese archivo.


```yaml
name: Despliegue en Cloud Run

on:
  workflow_dispatch:
  push:
    tags:
      - "v*"

permissions:
  contents: read

jobs:
  desplegar:
    runs-on: ubuntu-latest
    env:
      GCP_PROJECT_ID: ${{ vars.GCP_PROJECT_ID }}
      GCP_REGION: ${{ vars.GCP_REGION }}
      AR_REPOSITORY: ${{ vars.AR_REPOSITORY }}
      CLOUD_RUN_SERVICE: ${{ vars.CLOUD_RUN_SERVICE }}
    steps:
    - name: Checkout
      uses: actions/checkout@v7

    # Patrón transitorio de enseñanza (llave de cuenta de servicio).
    # Para federación de identidad, sustituir este paso por
    # workload_identity_provider y service_account, y añadir
    # permissions.id-token: write.
    - name: Autenticar en GCP
      uses: google-github-actions/auth@v3
      with:
        credentials_json: ${{ secrets.GCP_SA_KEY }}

    - name: Configurar gcloud
      uses: google-github-actions/setup-gcloud@v3

    - name: Autenticar Docker en Artifact Registry
      run: gcloud auth configure-docker "${GCP_REGION}-docker.pkg.dev" --quiet

    - name: Construir y publicar la imagen
      run: |
        IMAGEN="${GCP_REGION}-docker.pkg.dev/${GCP_PROJECT_ID}/${AR_REPOSITORY}/${CLOUD_RUN_SERVICE}:${GITHUB_SHA}"
        echo "IMAGE=${IMAGEN}" >> "${GITHUB_ENV}"
        docker build -t "${IMAGEN}" .
        docker push "${IMAGEN}"

    - name: Desplegar en Cloud Run
      run: |
        gcloud run deploy "${CLOUD_RUN_SERVICE}" \
          --project "${GCP_PROJECT_ID}" \
          --image "${IMAGE}" \
          --region "${GCP_REGION}" \
          --platform managed \
          --port 8080 \
          --allow-unauthenticated \
          --set-env-vars "APP_VERSION=${GITHUB_SHA}"
        gcloud run services describe "${CLOUD_RUN_SERVICE}" \
          --project "${GCP_PROJECT_ID}" \
          --region "${GCP_REGION}" \
          --format='value(status.url)'
```

Si el disparo fue una etiqueta, el *workflow* vigente también publica una etiqueta adicional con el nombre de esa etiqueta. El *workflow* comprueba, al inicio, que las cuatro variables del repositorio estén definidas.

Variables de repositorio previstas: ```GCP_PROJECT_ID```, ```GCP_REGION```, ```AR_REPOSITORY``` y ```CLOUD_RUN_SERVICE```. Secreto previsto: ```GCP_SA_KEY```. Ningún valor de estos se escribe en el repositorio.

## Actividad.

1. Recibir de quien imparte el curso el proyecto, la región y los nombres de registro y servicio, y cargarlos como variables del repositorio; cargar el secreto ```GCP_SA_KEY```.
2. Ejecutar el despliegue manual de comprobación y abrir la *URL* del servicio, comprobando ```GET /health```.
3. Disparar el *workflow* ```Despliegue en Cloud Run``` y verificar que la revisión nueva usa la imagen etiquetada con el *SHA* del *commit*, y que ```APP_VERSION``` en ```/health``` coincide con ese *SHA*.
4. Identificar, en el archivo del *workflow*, el único paso que habría que cambiar para pasar a federación de identidad.

## Mapa breve de servicios que no se practican aquí.

* **Cloud SQL.** Base de datos relacional gestionada. En cursos posteriores se conecta a la aplicación (a menudo mediante un *proxy* de autenticación, sin exponer el puerto de la base a internet).
* **Secret Manager.** Almacén de secretos de *GCP*. En cursos posteriores la aplicación lee de ahí contraseñas o llaves, en lugar de llevarlas en la imagen o en el repositorio.
* **Compute Engine.** Máquinas virtuales. El cuaderno 17 practica un servidor *Linux* como destino; no se aprovisiona esa máquina con *Compute Engine* en este curso.
* **Terraform.** Herramienta de infraestructura como código. Aquí los recursos se crean con ```gcloud``` para entender el destino; en cursos posteriores esos mismos destinos se declaran como código.

Saber que existen basta para este curso. No hace falta configurarlos en este laboratorio.

## Cierre.

El estudiante ya publicó una imagen en *Artifact Registry* y la sirvió en *Cloud Run* con un *workflow*. El cuaderno 17 hace el ejercicio equivalente sobre un servidor *Linux* alcanzado por *SSH*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023.</p>